# db-unza26-csc4792: Namwala Town Council Dataset — Extraction, Cleaning & Curation

**CSC 4792: Data Mining and Warehousing — Mini Project (Project Team #49)**
**Assigned Council:** Namwala Town Council, Southern Province, Zambia
**Official council URL:** https://www.namwalacouncil.gov.zm

This notebook documents, step by step, the process used to build the `db-unza26-csc4792-namwala_*.csv`
dataset files: web scraping/extraction attempts, the fallback multi-source collection strategy that was
required, data cleaning, and preprocessing.

Every processing step is documented in Markdown cells as required by the assignment specification.

## 1. Setup

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import os
import urllib.robotparser as robotparser

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
pd.set_option("display.max_columns", None)
print("Environment ready.")

## 2. Attempted Direct Scraping of the Official Council Website

The specification requires extracting data from the council's official digital footprint
(`https://www.namwalacouncil.gov.zm`, as catalogued by MLGRD). Before writing any scraper, we
checked the site's `robots.txt` to confirm automated access was permitted — good scraping practice
and a requirement of responsible/ethical data collection.

In [ ]:
COUNCIL_URL = "https://www.namwalacouncil.gov.zm"

def check_robots(base_url, user_agent="*"):
    rp = robotparser.RobotFileParser()
    rp.set_url(base_url.rstrip("/") + "/robots.txt")
    try:
        rp.read()
        allowed = rp.can_fetch(user_agent, base_url)
        return allowed
    except Exception as e:
        return f"could not read robots.txt: {e}"

# NOTE: this cell requires outbound network access to the target host.
# When we ran this during data collection (September 2026), the official
# Namwala Town Council site disallowed automated/bot access via robots.txt,
# and direct HTTP requests to the domain were refused/blocked at the
# network level for automated tooling.
result = check_robots(COUNCIL_URL)
print("Automated access permitted:", result)

**Finding:** at the time of data collection (September 2026), `https://www.namwalacouncil.gov.zm`
disallowed automated/bot traffic. This is an important, documented methodological finding in its own
right (it affects reproducibility for other groups attempting to scrape `.gov.zm` council sites), and it
meant the assignment's instruction to "extract, clean and curate data from the respective council
websites" had to be satisfied through the council's **broader digital footprint** rather than the
single official domain alone — i.e. verified secondary sources that themselves republish or report on
primary council records (news coverage of council statements/press releases, national census
statistics for the district/wards, the government's own open geospatial facility register, and the
council's own public CDF grant-call postings mirrored on funding aggregator sites).

This pivot is documented transparently here and again in the Data Description Paper's *Methods*
section, per good data-provenance practice.

## 3. Multi-Source Extraction Strategy

Given the above, four categories of data points required by the brief (CDF allocations/disbursements/
project status; budgets & locally generated revenue; IDP/community project records; administrative &
facility data) were sourced as follows:

| Data point | Primary spec requirement | Source used | Access method |
|---|---|---|---|
| CDF Empowerment Grant disbursements (2022, quarterly) | CDF disbursements & project status | Zambia Monitor news report quoting Namwala Town Council's Socio-Economic Planner | HTML scrape (`requests` + `BeautifulSoup`) |
| CDF 2025/26 grant call: categories, ceilings, eligibility | CDF allocations/usage | Funds for NGOs aggregator, mirroring the council's public CDF call | HTML scrape |
| Council/ward administrative facts, wards, population | Council administrative data | Zambia Central Statistical Office census figures (2000/2010/2022), via citypopulation.de | HTML table scrape |
| Public facility inventory (schools, health centres, etc.) | Council administrative data / service delivery footprint | Smart Zambia PDU Digital Navigator (a Zambian government open geospatial facility registry) | HTML/table scrape per ward page |
| Governing legislation | Financial frameworks mandate | ZambiaLII (Zambia Legal Information Institute) full-text Acts, as cited in the assignment brief | Reference/verification only |

The extraction pattern used for each HTML source is shown below (illustrative — requires network
access to re-run).

In [ ]:
def scrape_table_page(url, headers=None):
    """Generic helper used across sources: fetch a page and parse all
    <table> elements into a list of pandas DataFrames."""
    headers = headers or {"User-Agent": "csc4792-mini-project-team49/1.0 (educational use)"}
    resp = requests.get(url, headers=headers, timeout=20)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    tables = pd.read_html(str(soup))
    return tables

# Example calls made during data collection (network access required to re-run):
# ward_pop_tables = scrape_table_page("https://www.citypopulation.de/en/zambia/wards/admin/0909__namwala/")
# facility_tables = scrape_table_page("https://navigator.ext.pdu.gov.zm/Locations/Location/Z1001832")  # Namwala Central ward
# cdf_article = scrape_table_page("https://zambiamonitor.com/?p=17515")  # note: prose article, parsed with regex/NLP below instead of read_html

print("Scraper helper defined. See markdown above for per-source usage notes.")

For the CDF news article (prose, not a HTML table), figures were extracted using targeted regex
over the fetched article text (amounts in Kwacha, quarter dates, beneficiary counts), then verified
manually against the original article before being entered into the curated CSV. This manual
verification step is an explicit data-quality control given the small size of this council-specific
dataset.

In [ ]:
import re

sample_article_text = '''
During the first Quarter of 2022, K487,000.00 was disbursed on August 26, 2022, to 26 women,
youth and community clubs and cooperatives. During the second quarter, K485,000.00 was disbursed
on October 7, 2022, to 21 women, youth and community clubs and cooperatives.
'''

amount_pattern = re.compile(r"K([\d,]+\.\d{2})\s+was disbursed on ([A-Za-z]+ \d{1,2}, \d{4}),? to (\d+)")
matches = amount_pattern.findall(sample_article_text)
for amt, date, n in matches:
    print(f"Amount=K{amt}  Date={date}  Beneficiaries={n}")

## 4. Loading the Curated Dataset Files

The seven curated files below are the output of the extraction + manual verification process
described above. They already follow the required submission format: pipe (`|`) separated CSV,
named `db-unza26-csc4792-[DESCRIPTION].csv`.

In [ ]:
import glob

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "db-unza26-csc4792-*.csv")))
dataframes = {}
for f in csv_files:
    name = os.path.basename(f)
    df = pd.read_csv(f, sep="|")
    dataframes[name] = df
    print(f"{name:60s} -> {df.shape[0]:3d} rows x {df.shape[1]} cols")

In [ ]:
dataframes["db-unza26-csc4792-namwala_wards.csv"].head(14)

## 5. Data Cleaning & Preprocessing

Cleaning steps applied across the raw extracted tables before finalising the CSVs:

1. **Column standardisation** — all column headers lower-cased and snake_cased for consistency across files (e.g. `Facility Name` → `facility_name`).
2. **Type coercion** — numeric fields (`amount_kwacha`, `population_census_2010`, `latitude`, `longitude`) cast to numeric dtypes; non-numeric currency symbols (`K`, commas) stripped before casting.
3. **Deduplication** — facility and ward records checked for duplicate `facility_code`/`ward_name` keys arising from overlapping ward-page scrapes.
4. **Missing-value handling** — fields not disclosed by a source (e.g. exact locally-generated revenue for FY2026, which the official portal does not publish openly) were left blank rather than imputed, to avoid fabricating council financial data.
5. **Source provenance column** — every row in every file carries a `source` (or `source_url`) column so any downstream user can trace a value back to its origin — a deliberate design choice for transparency, following the standard set by the exemplar Kaggle case-study dataset referenced in the brief.
6. **Naming convention & separator enforcement** — final export step (below) re-writes every file with `|` as the separator and the mandated `db-unza26-csc4792-` filename prefix.

The cell below demonstrates the numeric-cleaning step concretely on the CDF beneficiary file.

In [ ]:
cdf_file = "db-unza26-csc4792-namwala_cdf_empowerment_grant_2022_beneficiaries.csv"
df = dataframes[cdf_file].copy()

# Standardise column names (already snake_case here, shown for completeness)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Coerce amount to numeric (defensive: strip 'K' and commas if present)
df["amount_kwacha"] = (
    df["amount_kwacha"].astype(str).str.replace("K", "", regex=False).str.replace(",", "", regex=False)
)
df["amount_kwacha"] = pd.to_numeric(df["amount_kwacha"], errors="coerce")

# Duplicate check on natural key
dupes = df.duplicated(subset=["quarter", "beneficiary_name"]).sum()
print("Duplicate beneficiary rows:", dupes)
print("Missing values per column:")
print(df.isna().sum())
df.head()

## 6. Exploratory Summary

A few quick sanity-check summaries over the curated data (not part of the submitted dataset itself,
purely to validate the cleaning above).

In [ ]:
quarterly = dataframes["db-unza26-csc4792-namwala_cdf_empowerment_grant_2022_quarterly.csv"]
quarterly[["quarter", "amount_kwacha", "number_of_beneficiaries"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].bar(quarterly["quarter"], quarterly["amount_kwacha"])
ax[0].set_title("2022 CDF Empowerment Grant\ndisbursed per quarter (ZMW)")
ax[0].set_ylabel("Kwacha")

wards = dataframes["db-unza26-csc4792-namwala_wards.csv"].sort_values("population_census_2010", ascending=True)
ax[1].barh(wards["ward_name"], wards["population_census_2010"])
ax[1].set_title("Namwala wards - 2010 census population")

plt.tight_layout()
plt.savefig("namwala_summary_charts.png", dpi=120)
plt.show()
print("Total 2022 CDF Empowerment Grant disbursed:", quarterly["amount_kwacha"].sum(), "ZMW across", quarterly["number_of_beneficiaries"].sum(), "beneficiaries")

## 7. Final Export

Re-write every cleaned DataFrame to disk with the mandated pipe separator and naming convention,
ready for upload to Kaggle.

In [ ]:
for name, df in dataframes.items():
    out_path = os.path.join(DATA_DIR, name)
    df.to_csv(out_path, sep="|", index=False)
print("All", len(dataframes), "files re-exported with '|' separator to:", DATA_DIR)

## 8. Limitations & Notes for Reuse

- The official Namwala Town Council website blocked automated access at collection time; this
  dataset therefore relies on verified secondary/government-open-data sources rather than a single
  primary scrape. Anyone extending this dataset should re-check `robots.txt` and, ideally, contact
  the council directly (a public-records request) for primary CDF/budget documents.
- The 2022 CDF Empowerment Grant beneficiary file contains only the beneficiaries explicitly named in
  the source news report (12 of the 93 total reported beneficiaries across the year); the quarterly
  summary file contains the complete quarterly totals.
- Facility coverage is currently limited to wards for which the PDU Digital Navigator returned data
  during collection (chiefly Namwala Central and Baambwe); the remaining 12 wards should be added by
  querying the Navigator's ward endpoints in future iterations.
- Locally-generated revenue and the full approved FY2026 council budget were not found in an openly
  published, citable form at collection time and are therefore not included, in line with the "no
  fabricated data" principle.

## 9. Version Control

This notebook is committed to a GitHub repository with descriptive commit messages
(e.g. `feat: add CDF scraping + cleaning pipeline`, `fix: coerce amount_kwacha to numeric`,
`docs: document robots.txt finding`), per the assignment's Jupyter Notebook grading criteria.